# 03 · Gold: business aggregates
Builds the reporting tables from the star schema (facts joined to dimensions). Run after `02_build_facts`.

| Table | Grain |
| --- | --- |
| `agg_monthly_sales` | month, with month-over-month growth |
| `agg_sales_by_product_category` | category + subcategory |
| `agg_sales_by_territory` | territory group + country + region |
| `agg_reseller_performance` | reseller, with revenue rank |
| `agg_salesperson_target_attainment` | salesperson + target month |

The revenue in each aggregate is checked against `fact_sales` before anything is written.

In [0]:
%run ../../config/pipeline_config

In [0]:
%run ../../utils/common_functions

In [0]:
def gold(name):
    return spark.table(f"{GOLD_SCHEMA}.{name}")

# Fact joined to its dimensions (inner joins are safe: 02_build_facts guarantees every key resolves)
sales = (gold("fact_sales")
    .join(gold("dim_date").select(F.col("DateKey").alias("OrderDateKey"), "Year", "YearMonth", "MonthStartDate"), "OrderDateKey")
    .join(gold("dim_product").select("ProductKey", "Category", "Subcategory"), "ProductKey")
    .join(gold("dim_sales_territory"), "SalesTerritoryKey")
    .join(gold("dim_reseller").select("ResellerKey", "ResellerName", "BusinessType", "CountryRegion"), "ResellerKey"))

def kpis():
    return [F.round(F.sum("SalesAmount"), 2).alias("TotalRevenue"),
            F.round(F.sum("CostAmount"), 2).alias("TotalCost"),
            F.round(F.sum("ProfitAmount"), 2).alias("TotalProfit"),
            F.countDistinct("SalesOrderNumber").alias("TotalOrders"),
            F.sum("Quantity").alias("TotalQuantity")]

def add_ratios(df):
    return (df.withColumn("ProfitMarginPct", F.round(F.col("TotalProfit") * 100 / F.col("TotalRevenue"), 2))
              .withColumn("AvgOrderValue", F.round(F.col("TotalRevenue") / F.col("TotalOrders"), 2)))

w_month = Window.orderBy("MonthStartDate")
agg_monthly_sales = (add_ratios(sales.groupBy("Year", "YearMonth", "MonthStartDate").agg(*kpis()))
    .withColumn("PrevMonthRevenue", F.lag("TotalRevenue").over(w_month))
    .withColumn("MoMGrowthPct", F.round((F.col("TotalRevenue") - F.col("PrevMonthRevenue")) * 100 / F.col("PrevMonthRevenue"), 2)))

agg_sales_by_product_category = add_ratios(sales.groupBy("Category", "Subcategory").agg(*kpis()))

agg_sales_by_territory = add_ratios(sales.groupBy("TerritoryGroup", "Country", "Region").agg(*kpis()))

agg_reseller_performance = (add_ratios(sales.groupBy("ResellerKey", "ResellerName", "BusinessType", "CountryRegion").agg(*kpis()))
    .withColumn("RevenueRank", F.dense_rank().over(Window.orderBy(F.desc("TotalRevenue")))))

actual_by_rep_month = (gold("fact_sales")
    .groupBy("EmployeeKey", F.trunc("OrderDate", "month").alias("TargetMonth"))
    .agg(F.round(F.sum("SalesAmount"), 2).alias("ActualRevenue")))

agg_salesperson_target_attainment = (gold("fact_sales_target")
    .join(gold("dim_salesperson").select("EmployeeKey", "SalespersonName", "Title"), "EmployeeKey")
    .join(actual_by_rep_month, ["EmployeeKey", "TargetMonth"], "left")
    .select("EmployeeKey", "EmployeeID", "SalespersonName", "Title", "TargetMonthKey", "TargetMonth", "TargetAmount",
            F.coalesce("ActualRevenue", F.lit(0.0)).alias("ActualRevenue"))
    .withColumn("AttainmentPct", F.round(F.col("ActualRevenue") * 100 / F.col("TargetAmount"), 2))
    .withColumn("TargetMet", F.col("ActualRevenue") >= F.col("TargetAmount")))

aggregates = {
    "agg_monthly_sales": agg_monthly_sales,
    "agg_sales_by_product_category": agg_sales_by_product_category,
    "agg_sales_by_territory": agg_sales_by_territory,
    "agg_reseller_performance": agg_reseller_performance,
    "agg_salesperson_target_attainment": agg_salesperson_target_attainment,
}

In [0]:
fact_revenue = gold("fact_sales").agg(F.round(F.sum("SalesAmount"), 2)).first()[0]

dq = DataQualityChecker("gold")
for name in ["agg_monthly_sales", "agg_sales_by_product_category", "agg_sales_by_territory", "agg_reseller_performance"]:
    agg_revenue = aggregates[name].agg(F.round(F.sum("TotalRevenue"), 2)).first()[0]
    dq.check(name, f"TotalRevenue reconciles to fact_sales ({fact_revenue})", 0 if abs(agg_revenue - fact_revenue) < 1 else 1)
dq.check("agg_salesperson_target_attainment", "one row per target",
         abs(agg_salesperson_target_attainment.count() - gold("fact_sales_target").count()))
display(dq.results_df())
dq.assert_no_critical_failures()

for name, df in aggregates.items():
    write_delta_table(df.withColumn("gold_load_timestamp", F.current_timestamp()),
                      f"{GOLD_SCHEMA}.{name}", f"{GOLD_PATH}{name}")